In [1]:

!nvidia-smi
!pip install -q numba cupy-cuda12x pytest scipy

import numpy as np
import numba
from numba import cuda
import cupy as cp
import scipy

print("NumPy:", np.__version__)
print("Numba:", numba.__version__)
print("CuPy:", cp.__version__)
print("CUDA available:", cuda.is_available())

if not cuda.is_available():
    raise RuntimeError(
        "GPU не обнаружен. Выбери Runtime → Change runtime type → GPU."
    )

print("GPU:", cuda.get_current_device().name)


Thu Oct 15 09:40:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

/usr/local/lib/python3.13/dist-packages/cupy/_environment.py:677: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy-cuda12x, cupy-cuda13x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


NumPy: 2.1.3
Numba: 0.61.2
CuPy: 14.2.0
CUDA available: True
GPU: Tesla T4


In [2]:

from numba import cuda
import numpy as np

@cuda.jit
def vector_add_1d(a, b, out):
    i = cuda.grid(1)
    if i < out.size:
        out[i] = a[i] + b[i]

N = 1000003
a = np.random.rand(N).astype(np.float32)
b = np.random.rand(N).astype(np.float32)
out = np.empty_like(a)

threads = 256
blocks = (N + threads - 1) // threads

vector_add_1d[blocks, threads](a, b, out)

np.testing.assert_allclose(out, a + b, atol=1e-5)
print("Task 1 PASSED")
print("First 5 results:", out[:5])


Task 1 PASSED
First 5 results: [1.0643094 1.7961146 1.3419371 1.4920182 0.7675656]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [3]:

@cuda.jit
def vector_scale_grid_stride(x, alpha, out):
    i = cuda.grid(1)
    stride = cuda.gridsize(1)

    while i < x.size:
        out[i] = x[i] * alpha
        i += stride

N = 1000000
alpha = np.float32(2.5)
x = np.random.rand(N).astype(np.float32)
out = np.empty_like(x)

vector_scale_grid_stride[64, 256](x, alpha, out)

np.testing.assert_allclose(out, x * alpha, atol=1e-5)
print("Task 2 PASSED")
print("First 5 results:", out[:5])


Task 2 PASSED
First 5 results: [0.19230814 2.0500932  0.40660885 0.21318564 1.4312465 ]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 64 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [4]:

@cuda.jit
def matrix_add_2d(A, B, C):
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x

    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = A[row, col] + B[row, col]

M, N = 513, 1021

A = np.random.rand(M, N).astype(np.float32)
B = np.random.rand(M, N).astype(np.float32)
C = np.empty_like(A)

threads = (16, 16)
blocks = ((N + 15) // 16, (M + 15) // 16)

matrix_add_2d[blocks, threads](A, B, C)

np.testing.assert_allclose(C, A + B, atol=1e-5)
print("Task 3 PASSED")
print("Matrix shape:", C.shape)


Task 3 PASSED
Matrix shape: (513, 1021)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [5]:

@cuda.jit
def matrix_transpose_naive(A, B):
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x

    if row < A.shape[0] and col < A.shape[1]:
        B[col, row] = A[row, col]

M, N = 257, 513

A = np.random.rand(M, N).astype(np.float32)
B = np.empty((N, M), dtype=np.float32)

threads = (16, 16)
blocks = ((N + 15) // 16, (M + 15) // 16)

matrix_transpose_naive[blocks, threads](A, B)

np.testing.assert_allclose(B, A.T, atol=1e-5)
print("Task 4 PASSED")
print("Original shape:", A.shape)
print("Transposed shape:", B.shape)


Task 4 PASSED
Original shape: (257, 513)
Transposed shape: (513, 257)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [6]:

@cuda.jit
def volume_init_3d(vol, D, H, W):
    x = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    y = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    z = cuda.blockIdx.z * cuda.blockDim.z + cuda.threadIdx.z

    if z < D and y < H and x < W:
        vol[z, y, x] = z * 10000 + y * 100 + x

D, H, W = 33, 47, 65
vol = np.empty((D, H, W), dtype=np.int32)

threads = (8, 8, 8)
blocks = (
    (W + 7) // 8,
    (H + 7) // 8,
    (D + 7) // 8
)

volume_init_3d[blocks, threads](vol, D, H, W)

z, y, x = np.indices((D, H, W))
expected = z * 10000 + y * 100 + x

np.testing.assert_array_equal(vol, expected)
print("Task 5 PASSED")
print("Volume shape:", vol.shape)
print("Example vol[2, 3, 4]:", vol[2, 3, 4])


Task 5 PASSED
Volume shape: (33, 47, 65)
Example vol[2, 3, 4]: 20304


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [7]:

@cuda.jit
def rgb_to_grayscale(img_rgb, img_gray):
    y, x = cuda.grid(2)

    if y < img_rgb.shape[0] and x < img_rgb.shape[1]:
        r = img_rgb[y, x, 0]
        g = img_rgb[y, x, 1]
        b = img_rgb[y, x, 2]

        img_gray[y, x] = (
            0.299 * r + 0.587 * g + 0.114 * b
        )

H, W = 256, 256

img_rgb = np.random.randint(
    0, 256, (H, W, 3), dtype=np.uint8
)
img_gray = np.empty((H, W), dtype=np.float32)

threads = (16, 16)
blocks = ((H + 15) // 16, (W + 15) // 16)

rgb_to_grayscale[blocks, threads](img_rgb, img_gray)

expected = (
    0.299 * img_rgb[:, :, 0].astype(np.float32)
    + 0.587 * img_rgb[:, :, 1].astype(np.float32)
    + 0.114 * img_rgb[:, :, 2].astype(np.float32)
)

np.testing.assert_allclose(img_gray, expected, atol=1e-4)
print("Task 6 PASSED")
print("Image shape:", img_gray.shape)


Task 6 PASSED
Image shape: (256, 256)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [8]:

@cuda.jit
def vector_lerp(a, b, t, out):
    i = cuda.grid(1)

    if i < out.size:
        out[i] = (1.0 - t[i]) * a[i] + t[i] * b[i]

N = 100000
a = np.random.rand(N).astype(np.float32)
b = np.random.rand(N).astype(np.float32)
t = np.random.rand(N).astype(np.float32)
out = np.empty_like(a)

threads = 256
blocks = (N + threads - 1) // threads

vector_lerp[blocks, threads](a, b, t, out)

expected = (1 - t) * a + t * b
np.testing.assert_allclose(out, expected, atol=1e-5)

print("Task 7 PASSED")
print("First 5 results:", out[:5])


Task 7 PASSED
First 5 results: [0.3251803  0.7561695  0.35867336 0.8027532  0.11803547]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [9]:

@cuda.jit
def vector_threshold(x, thresh, mask):
    i = cuda.grid(1)
    stride = cuda.gridsize(1)

    while i < x.size:
        if x[i] >= thresh:
            mask[i] = 1
        else:
            mask[i] = 0

        i += stride

N = 100000
x = np.random.rand(N).astype(np.float32)
thresh = np.float32(0.5)
mask = np.empty(N, dtype=np.uint8)

vector_threshold[64, 256](x, thresh, mask)

expected = (x >= thresh).astype(np.uint8)
np.testing.assert_array_equal(mask, expected)

print("Task 8 PASSED")
print("Number of ones:", int(mask.sum()))


Task 8 PASSED
Number of ones: 50003


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 64 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [10]:

@cuda.jit
def benchmark_strided_access(in_arr, out_arr, stride):
    i = cuda.grid(1)

    if i < out_arr.size:
        idx = i * stride
        if idx < in_arr.size:
            out_arr[i] = in_arr[idx]

N = 1_000_000
x = np.arange(N, dtype=np.float32)

threads = 256

for stride in (1, 2, 4, 8, 16, 32):
    size = (N + stride - 1) // stride
    out = np.empty(size, dtype=np.float32)
    blocks = (size + threads - 1) // threads

    # Warm-up: compile and run the kernel
    benchmark_strided_access[blocks, threads](x, out, stride)
    cuda.synchronize()

    start = cuda.event()
    end = cuda.event()

    start.record()
    benchmark_strided_access[blocks, threads](x, out, stride)
    end.record()
    end.synchronize()

    elapsed_ms = cuda.event_elapsed_time(start, end)

    np.testing.assert_array_equal(out, x[::stride])
    print(
        f"Stride={stride:2d} | "
        f"Time={elapsed_ms:.4f} ms | "
        f"Output={len(out)} elements"
    )

print("Task 9 correctness PASSED")


Stride= 1 | Time=3.8812 ms | Output=1000000 elements
Stride= 2 | Time=2.6676 ms | Output=500000 elements
Stride= 4 | Time=2.1708 ms | Output=250000 elements
Stride= 8 | Time=1.9758 ms | Output=125000 elements
Stride=16 | Time=1.8430 ms | Output=62500 elements
Stride=32 | Time=1.7522 ms | Output=31250 elements
Task 9 correctness PASSED


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 123 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [11]:

@cuda.jit
def stencil_1d_shared(in_arr, out_arr, radius):
    shared = cuda.shared.array(shape=320, dtype=numba.float32)

    tid = cuda.threadIdx.x
    i = cuda.blockIdx.x * cuda.blockDim.x + tid

    n = in_arr.size
    tile_size = cuda.blockDim.x + 2 * radius

    # Cooperative loading of the tile and halo
    for j in range(tid, tile_size, cuda.blockDim.x):
        idx = cuda.blockIdx.x * cuda.blockDim.x + j - radius

        if idx < 0:
            idx = 0
        elif idx >= n:
            idx = n - 1

        shared[j] = in_arr[idx]

    cuda.syncthreads()

    if i < n:
        total = 0.0

        for k in range(2 * radius + 1):
            total += shared[tid + k]

        out_arr[i] = total / (2 * radius + 1)


N = 10000
radius = 2

x = np.random.rand(N).astype(np.float32)
out = np.empty_like(x)

threads = 256
blocks = (N + threads - 1) // threads

stencil_1d_shared[blocks, threads](x, out, radius)

# Reference: nearest boundary, matching the task specification
from scipy.ndimage import uniform_filter1d

expected = uniform_filter1d(
    x, size=2 * radius + 1, mode="nearest"
)

np.testing.assert_allclose(out, expected, atol=1e-4)

print("Task 10 PASSED")
print("First 5 results:", out[:5])


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Task 10 PASSED
First 5 results: [0.7211147  0.6040433  0.59588194 0.48627052 0.48589155]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
